# 02. ?? Baseline ?? ??

> ???: 2026-07-06 KST  
> ??: ?? `baseline.ipynb`? ??? ??, feature ??, RandomForest ?? ??? ???? 2024 holdout?? local score ???? ???.

?? ???? ????? ?????. ?? CSV, ?? ??, output ???? ???? ???. ?? baseline? ??? ??? ???? ?? ???, ?? `train.py`/`inference.py` ?? ???? ??? ?? ??? ??? ????? ??? ????.


## Decision Box 0: ?? ???? ??

| ?? | ?? | ?? |
|---|---|---|
| ?? ?? | `.ipynb` | ????? ??? ????? ???? ?? ??? ???. |
| ?? | ?? RandomForest baseline? ?? | LightGBM/CatBoost ? ?? ??? ?? ???? ???. |
| ?? ??? | ???? ?? | ?? baseline ?? ????? ?? ?? ???? ????. |
| SCADA feature | ???? ?? | ?? baseline? LDAPS/GFS ?? ??? calendar feature? ????. |
| ?? ?? | ???? ?? | ?? ??? ??? ??? local ???? ?? ?? ????. |
| ?? | 2024 time holdout | random split? ??? 2025 ?? ??? ??? 1? holdout?? ??. |


In [1]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
  sys.path.insert(0, str(SRC_DIR))

from baram.metrics import CAPACITY_KWH, TARGET_COLS, metric

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "open"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
OFFICIAL_BASELINE_NOTEBOOK = PROJECT_ROOT / "references" / "official" / "notebooks" / "baseline.ipynb"

RANDOM_STATE = 42
OFFICIAL_RF_PARAMS = {
  "n_estimators": 120,
  "max_depth": 14,
  "min_samples_leaf": 8,
  "max_features": "sqrt",
  "random_state": RANDOM_STATE,
  "n_jobs": -1,
}

print("project:", PROJECT_ROOT)
print("data exists:", DATA_DIR.exists())
print("target columns:", TARGET_COLS)
print("capacity:", CAPACITY_KWH)


project: C:\Users\kik32\workspace\Dacon\2026-BARAM-Wind-Power-Prediction-AI-Competition
data exists: True
target columns: ['kpx_group_1', 'kpx_group_2', 'kpx_group_3']
capacity: {'kpx_group_1': 21600, 'kpx_group_2': 21600, 'kpx_group_3': 21000}


## 1. ?? baseline ?? ?? ??

?? baseline? ?? ???.

1. `train_labels`, `sample_submission`, LDAPS/GFS train/test CSV ??
2. LDAPS/GFS? `forecast_kst_dtm` ???? ?? ?? ??
3. calendar feature ??
4. `SimpleImputer(strategy="median")` ??
5. target? non-null label ?? ??? `RandomForestRegressor` ?? ??
6. test ?? ? `0 ~ capacity` clipping
7. `baseline_submit.csv` ??

?? ?????? 7? ??? ?? ??, 2024 holdout? ??? local score? ????.


In [2]:
officialNotebook = json.loads(OFFICIAL_BASELINE_NOTEBOOK.read_text(encoding="utf-8"))
codeCells = ["".join(cell.get("source", [])) for cell in officialNotebook["cells"] if cell.get("cell_type") == "code"]

pd.DataFrame(
  {
    "code_cell": list(range(len(codeCells))),
    "line_count": [len(code.splitlines()) for code in codeCells],
    "first_line": [code.splitlines()[0] if code.splitlines() else "" for code in codeCells],
  }
)


,code_cell,line_count,first_line
0,0,6,from pathlib import Path
1,1,26,"DATA_DIR = Path(""."")"
2,2,49,"def aggregate_weather(df, prefix):"
3,3,25,"imputer = SimpleImputer(strategy=""median"")"
4,4,8,"submission = sample_submission[[""forecast_id"",..."
5,5,3,"output_path = DATA_DIR / ""baseline_submit.csv"""


## 2. ?? ??? ??? ??? ??

?? CSV/XLSX? Git? ??? ?? `data/raw/open/`? ???? ??. ? ???? ?? ??? shape? ????, ?? ??? ???? ???.


In [3]:
requiredFiles = [
  TRAIN_DIR / "train_labels.csv",
  DATA_DIR / "sample_submission.csv",
  TRAIN_DIR / "ldaps_train.csv",
  TRAIN_DIR / "gfs_train.csv",
  TEST_DIR / "ldaps_test.csv",
  TEST_DIR / "gfs_test.csv",
]
missingFiles = [path for path in requiredFiles if not path.exists()]
if missingFiles:
  raise FileNotFoundError("?? ?? ??? ??? ????: " + ", ".join(str(path) for path in missingFiles))

trainLabels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
sampleSubmission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
ldapsTrain = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfsTrain = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
ldapsTest = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
gfsTest = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")

trainLabels["kst_dtm"] = pd.to_datetime(trainLabels["kst_dtm"])
sampleSubmission["forecast_kst_dtm"] = pd.to_datetime(sampleSubmission["forecast_kst_dtm"])

inventory = pd.DataFrame(
  [
    ("train_labels", trainLabels.shape[0], trainLabels.shape[1], str(trainLabels["kst_dtm"].min()), str(trainLabels["kst_dtm"].max())),
    ("sample_submission", sampleSubmission.shape[0], sampleSubmission.shape[1], str(sampleSubmission["forecast_kst_dtm"].min()), str(sampleSubmission["forecast_kst_dtm"].max())),
    ("ldaps_train", ldapsTrain.shape[0], ldapsTrain.shape[1], "", ""),
    ("gfs_train", gfsTrain.shape[0], gfsTrain.shape[1], "", ""),
    ("ldaps_test", ldapsTest.shape[0], ldapsTest.shape[1], "", ""),
    ("gfs_test", gfsTest.shape[0], gfsTest.shape[1], "", ""),
  ],
  columns=["file", "rows", "cols", "start", "end"],
)
inventory


,file,rows,cols,start,end
0,train_labels,26304,4,2022-01-01 01:00:00,2025-01-01 00:00:00
1,sample_submission,8760,5,2025-01-01 01:00:00,2026-01-01 00:00:00
2,ldaps_train,420864,35,,
3,gfs_train,236736,40,,
4,ldaps_test,140160,35,,
5,gfs_test,78840,40,,


In [4]:
labelAuditRows = []
for target in TARGET_COLS:
  capacity = CAPACITY_KWH[target]
  series = trainLabels[target]
  labelAuditRows.append(
    {
      "target": target,
      "capacity": capacity,
      "non_null": int(series.notna().sum()),
      "missing": int(series.isna().sum()),
      "min": float(series.min(skipna=True)),
      "max": float(series.max(skipna=True)),
      "capacity_exceed": int((series > capacity).sum()),
      "official_valid_candidate": int((series >= capacity * 0.10).sum()),
    }
  )

pd.DataFrame(labelAuditRows)


,target,capacity,non_null,missing,min,max,capacity_exceed,official_valid_candidate
0,kpx_group_1,21600,26200,104,0.0,21275.305,0,15915
1,kpx_group_2,21600,26201,103,0.0,21362.084,0,15891
2,kpx_group_3,21000,17538,8766,0.0,21130.674,38,9414


## Decision Box 1: ?? baseline?? ??? ?

| ?? | ?? ?? | ?? |
|---|---|---|
| LDAPS/GFS `forecast_kst_dtm`? ?? | ?? | ?? baseline? ?? ??? ?? ???? ??. |
| calendar sin/cos | ?? | ???? ???? ????? ????. |
| target? ?? ?? | ?? | Group 3 label ??? ???? target? non-null mask? ????. |
| median imputer | ?? | weather merge ? ??? ??? baseline ???? ????. |
| prediction clipping | metric ??? ?? | metric? ???, ???? baseline inference ??? ????. |
| random validation | ???? ?? | ??? ?? ?? ??? 2024 holdout? ????. |


## 3. ?? feature ?? ?? ??

?? ??? ?? baseline? `aggregate_weather`, `calendar_features` ??? ????. ?? ??? ?? ?? ???.


In [5]:
def aggregateWeather(df, prefix):
  frame = df.copy()
  frame["forecast_kst_dtm"] = pd.to_datetime(frame["forecast_kst_dtm"])
  dropCols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
  valueCols = [col for col in frame.columns if col not in {"forecast_kst_dtm", *dropCols}]
  aggregated = frame.groupby("forecast_kst_dtm")[valueCols].mean()
  aggregated.columns = [f"{prefix}_{col}_mean" for col in aggregated.columns]
  return aggregated.reset_index()


def calendarFeatures(dtSeries):
  dt = pd.to_datetime(dtSeries)
  out = pd.DataFrame(index=dt.index)
  out["month"] = dt.dt.month
  out["day"] = dt.dt.day
  out["hour"] = dt.dt.hour
  out["dayofweek"] = dt.dt.dayofweek
  out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
  out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
  out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
  out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
  out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
  return out


def buildFeatureFrame(labelDf, ldapsDf, gfsDf, sampleDf=None):
  weather = aggregateWeather(ldapsDf, "ldaps").merge(
    aggregateWeather(gfsDf, "gfs"), on="forecast_kst_dtm", how="inner"
  )

  if labelDf is not None:
    base = labelDf.rename(columns={"kst_dtm": "forecast_kst_dtm"})
    merged = base.merge(weather, on="forecast_kst_dtm", how="left")
    X = pd.concat(
      [calendarFeatures(merged["forecast_kst_dtm"]), merged.drop(columns=["forecast_kst_dtm", *TARGET_COLS])],
      axis=1,
    )
    return merged, X, weather

  merged = sampleDf[["forecast_id", "forecast_kst_dtm"]].merge(weather, on="forecast_kst_dtm", how="left")
  X = pd.concat(
    [calendarFeatures(merged["forecast_kst_dtm"]), merged.drop(columns=["forecast_id", "forecast_kst_dtm"])],
    axis=1,
  )
  return merged, X, weather


trainDf, XTrain, trainWeather = buildFeatureFrame(trainLabels, ldapsTrain, gfsTrain)
testDf, XTest, testWeather = buildFeatureFrame(None, ldapsTest, gfsTest, sampleSubmission)

featureSummary = pd.DataFrame(
  [
    ("train_weather", trainWeather.shape[0], trainWeather.shape[1], int(trainWeather.isna().sum().sum())),
    ("test_weather", testWeather.shape[0], testWeather.shape[1], int(testWeather.isna().sum().sum())),
    ("train_df", trainDf.shape[0], trainDf.shape[1], int(trainDf.isna().sum().sum())),
    ("test_df", testDf.shape[0], testDf.shape[1], int(testDf.isna().sum().sum())),
    ("X_train", XTrain.shape[0], XTrain.shape[1], int(XTrain.isna().sum().sum())),
    ("X_test", XTest.shape[0], XTest.shape[1], int(XTest.isna().sum().sum())),
  ],
  columns=["name", "rows", "cols", "missing_cells"],
)
featureSummary


,name,rows,cols,missing_cells
0,train_weather,26304,66,0
1,test_weather,8760,66,47
2,train_df,26304,69,8973
3,test_df,8760,67,47
4,X_train,26304,74,0
5,X_test,8760,74,47


## 4. 2024 holdout local score

?? baseline ????? validation? ??. ???? ?? feature? ?? RandomForest ????? ??, 2022~2023? ???? 2024? ????. Group 3? 2022 label? ???? target? non-null mask? ???? 2023 ?? ??? ???.


In [6]:
def trainAndPredictByTarget(XTrainLocal, trainLocalDf, XPredictLocal, modelParams):
  imputer = SimpleImputer(strategy="median")
  XTrainImp = pd.DataFrame(imputer.fit_transform(XTrainLocal), columns=XTrainLocal.columns, index=XTrainLocal.index)
  XPredictImp = pd.DataFrame(imputer.transform(XPredictLocal), columns=XPredictLocal.columns, index=XPredictLocal.index)

  predictions = pd.DataFrame(index=XPredictLocal.index)
  trainRows = {}
  elapsedRows = {}

  for target in TARGET_COLS:
    startedAt = time.time()
    trainMask = trainLocalDf[target].notna()
    yTrain = trainLocalDf.loc[trainMask, target]

    model = RandomForestRegressor(**modelParams)
    model.fit(XTrainImp.loc[trainMask], yTrain)

    pred = model.predict(XPredictImp)
    predictions[target] = np.clip(pred, 0, CAPACITY_KWH[target])
    trainRows[target] = int(trainMask.sum())
    elapsedRows[target] = round(time.time() - startedAt, 2)

  return predictions, trainRows, elapsedRows


trainYears = trainDf["forecast_kst_dtm"].dt.year
localTrainMask = trainYears < 2024
localValidMask = trainYears == 2024

XLocalTrain = XTrain.loc[localTrainMask].copy()
trainLocalDf = trainDf.loc[localTrainMask].copy()
XLocalValid = XTrain.loc[localValidMask].copy()
validDf = trainDf.loc[localValidMask, ["forecast_kst_dtm", *TARGET_COLS]].copy()

validPred, trainRows, elapsedRows = trainAndPredictByTarget(
  XLocalTrain, trainLocalDf, XLocalValid, OFFICIAL_RF_PARAMS
)

scoreTuple = metric(validDf[TARGET_COLS], validPred[TARGET_COLS])
scoreSummary = pd.DataFrame(
  [
    ("total_score", scoreTuple[0]),
    ("one_minus_nmae", scoreTuple[1]),
    ("ficr", scoreTuple[2]),
  ],
  columns=["metric", "value"],
)

scoreSummary


,metric,value
0,total_score,0.576938
1,one_minus_nmae,0.863228
2,ficr,0.290647


In [7]:
diagnosticRows = []
for target in TARGET_COLS:
  capacity = CAPACITY_KWH[target]
  actual = validDf[target]
  pred = validPred[target]
  validMask = actual >= capacity * 0.10
  errorRate = (pred[validMask] - actual[validMask]).abs() / capacity
  diagnosticRows.append(
    {
      "target": target,
      "train_rows": trainRows[target],
      "valid_rows": int(actual.notna().sum()),
      "official_valid_rows": int(validMask.sum()),
      "elapsed_sec": elapsedRows[target],
      "mae_all": float((pred[actual.notna()] - actual[actual.notna()]).abs().mean()),
      "nmae_valid": float(errorRate.mean()),
      "within_6pct": float((errorRate <= 0.06).mean()),
      "within_8pct": float((errorRate <= 0.08).mean()),
      "pred_min": float(pred.min()),
      "pred_max": float(pred.max()),
    }
  )

pd.DataFrame(diagnosticRows)


,target,train_rows,valid_rows,official_valid_rows,elapsed_sec,mae_all,nmae_valid,within_6pct,within_8pct,pred_min,pred_max
0,kpx_group_1,17421,8778,4989,0.93,2215.631078,0.129808,0.283223,0.377430,224.646139,18894.488671
1,kpx_group_2,17422,8778,4976,1.04,2188.699786,0.128420,0.306069,0.401929,263.016646,19391.718817
2,kpx_group_3,8759,8778,4566,0.59,2175.441546,0.152087,0.260841,0.345817,164.088409,17407.855592


## Decision Box 2: baseline ???? ??? ??

| ?? | ?? | ?? ?? ?? |
|---|---|---|
| ?? baseline? validation? ?? | LB ?? ? local score ???? ?? ?? | train/inference ?? ?? validation scoreboard? ?? ?? |
| feature? ?? ????? | ?? ??, lead hour, ?? ? ??? ??? | P0 ??? `mean/std/min/max`? lead hour?? ???? |
| target? ??? ?? ??? ?? | Group 3 label ?? ??? ????? ???? | Group 3 ?? fold? pooled model? ??? ???? |
| clipping? metric ??? ??? inference ???? | official metric? ??? ??? ???? ?? | validator/??? ???? ??? capacity ??? ??? |


## 5. ?? ?? preview

?? ?? sample submission ??? inference feature shape? ????. ?? ?????? `baseline_submit.csv`? ???? ???.


In [8]:
submissionSchema = pd.DataFrame(
  {
    "column": sampleSubmission.columns,
    "dtype": [str(sampleSubmission[col].dtype) for col in sampleSubmission.columns],
    "missing": [int(sampleSubmission[col].isna().sum()) for col in sampleSubmission.columns],
  }
)

print("sample_submission rows:", len(sampleSubmission))
print("X_test shape:", XTest.shape)
submissionSchema


sample_submission rows: 8760
X_test shape: (8760, 74)


,column,dtype,missing
0,forecast_id,str,0
1,forecast_kst_dtm,datetime64[us],0
2,kpx_group_1,int64,0
3,kpx_group_2,int64,0
4,kpx_group_3,int64,0


## ?? ?? ??

| ?? | ?? | ?? |
|---|---|---|
| 2026-07-06 | baseline ?? ??? `.ipynb`? ?? | ????? ??? ????? ????? ?? ??? ?? |
| 2026-07-06 | ?? baseline feature/model ??? ?? ?? | ?? ?? ??? ?? ???? ???? local/LB ??? ??? ? ?? |
| 2026-07-06 | 2024 holdout score? baseline ????? ?? | ?? baseline?? validation? ???? ??? time split ??? ?? |
| 2026-07-06 | ?? CSV? ???? ?? | ?? ??? ??? ???? ???? LB ?? ??? ?? |

?? ?? ??? ?? baseline? `train.py`? `inference.py`? ????, config/seed/submission ledger ?? ??? ???? ???.
